In [1]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery

In [2]:
load_dotenv()

True

In [5]:
project_id = os.getenv("PROJECT_ID")
dataset = os.getenv("DATASET")
bucket_name = os.getenv("BUCKET_NAME")
cleaned_folder = os.getenv("CLEANED_FOLDER_NAME")

gs_folder = f"gs://{bucket_name}/{cleaned_folder}/date=2026-04-11/*.parquet"

In [15]:
hive_partitioning_options = bigquery.HivePartitioningOptions()
hive_partitioning_options.mode="AUTO"
hive_partitioning_options.source_uri_prefix=f"gs://{bucket_name}/{cleaned_folder}/"


job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        autodetect=True,
        write_disposition="WRITE_TRUNCATE",
        hive_partitioning=hive_partitioning_options
    )

In [16]:
def upload_to_bq(uri, project_id, dataset, job_config):
    client = bigquery.Client()

    staging_table_id = f"{project_id}.{dataset}.test_load_staging"

    load_job = client.load_table_from_uri(
        uri, staging_table_id, job_config=job_config
    )
    
    # Waits for the job to complete.
    load_job.result()

    # Get the staging table to check the number of rows loaded
    staging_table = client.get_table(staging_table_id)
    print("Loaded {} rows.".format(staging_table.num_rows))


In [17]:
upload_to_bq(gs_folder, project_id, dataset, job_config)

Loaded 29 rows.
